In [1]:
from useful.preprocess import *
from useful.train import *
from pathlib import Path
import pandas as pd

Opening Annotations

In [2]:
annotations_path = Path('../../data/annotations')
csvs_path = annotations_path.glob('text*.csv')
annotations = []

for csv_path in csvs_path:
    csv = pd.read_csv(csv_path)
    annotations.append(csv)

print(len(annotations))
reader = CSVReader(annotations)
print(reader.entities.columns)
print(reader.entities_relations.columns)


12
Index(['data', 'label'], dtype='str')
Index(['e1_type', 'e2_type', 'intermediate_text', 'relation_label'], dtype='str')


Training Models

In [3]:
entity_classifier = EntityClassifier(C=0.1)
entity_classifier.fit(reader.entities['data'], reader.entities['label'])
relation_classifier = RelationClassifier(C=0.1)
relation_classifier.fit(reader.entities_relations['e1_type'], reader.entities_relations['e2_type'], reader.entities_relations['intermediate_text'], reader.entities_relations['relation_label'])

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

Testing

In [ ]:
test = pd.Series(['the patient'])
ans = entity_classifier.predict(test)
print(ans)

0    PATIENT
dtype: str


In [ ]:
def process_cases_to_graph_csvs(
    cases_df: pd.DataFrame,
    entity_classifier: EntityClassifier,
    relation_classifier: RelationClassifier,
    output_graph_path: Path = Path("outputs/graph.csv"),
    max_distance: int = 10,
    context_k: int = 3
):
    """
    Lê o DataFrame de casos clínicos (cases.csv) e gera o arquivo graph.csv.
    """
    processor = MedicalTextProcessor()
    
    graph_list = []

    for _, case_row in cases_df.iterrows():
        case_id = case_row['case_id']
        raw_text = str(case_row['case_text'])

        # Divide o texto em períodos limpos
        sentences = processor.split_sentences(raw_text)

        global_token_offset = 0
        
        for sentence in sentences:
            # Localiza o offset do caractere do período no texto completo
            sent_char_start = raw_text.find(sentence)
            if sent_char_start == -1:
                sent_char_start = 0

            # Processa a frase extraindo o Doc, candidatos a entidades e pares
            doc, candidates, pairs = processor.process_sentence(
                sentence, 
                max_distance=max_distance, 
                context_k=context_k
            )

            if not candidates:
                global_token_offset += len(doc)
                continue

            # 1. Classificação de Entidades (Modelo 1)
            cand_series = pd.Series([c['text'] for c in candidates])
            predicted_types = entity_classifier.predict(cand_series)

            # Mapeia entidades válidas
            valid_entities_in_sent = {}

            for cand, p_type in zip(candidates, predicted_types):
                if p_type not in ['NOT_ENTITY', 'OUTSIDE']:
                    # Guarda objeto da entidade com o tipo para a fase de relações
                    cand['type'] = p_type
                    valid_entities_in_sent[cand['text']] = cand

            # 2. Classificação de Relações (Modelo 2)
            for pair in pairs:
                e1_text = pair['e1']['text']
                e2_text = pair['e2']['text']

                # Só avalia relação se ambas as entidades forem classificadas como válidas
                if e1_text in valid_entities_in_sent and e2_text in valid_entities_in_sent:
                    e1 = valid_entities_in_sent[e1_text]
                    e2 = valid_entities_in_sent[e2_text]

                    s_e1_type = pd.Series([e1['type']])
                    s_e2_type = pd.Series([e2['type']])
                    s_intermed = pd.Series([pair['intermediate_text']])

                    rel_label = relation_classifier.predict(s_e1_type, s_e2_type, s_intermed).iloc[0]

                    if rel_label != 'NO_RELATION':
                        graph_list.append({
                            'case_id': case_id,
                            'e1_normalizado': e1['text'].lower(),
                            'e1_text': e1['text'],
                            'e1_type': e1['type'],
                            'e2_normalizado': e2['text'].lower(),
                            'e2_text': e2['text'],
                            'e2_type': e2['type'],
                            'relation_label': rel_label,
                            'intermediate_text': pair['intermediate_text']
                        })

            # Atualiza o offset acumulado de tokens
            global_token_offset += len(doc)

    # Converte em DataFrame
    df_graph = pd.DataFrame(graph_list)

    # Garante a ordem exata das colunas especificadas no projeto
    graph_columns = [
        'edge_id', 'case_id', 'e1_normalizado', 'e1_text', 'e1_type',
        'e2_normalizado', 'e2_text', 'e2_type', 'relation_label', 'intermediate_text'
    ]

    df_graph = df_graph.reindex(columns=graph_columns)

    # Salva o arquivos CSV
    df_graph.to_csv(output_graph_path, index=False)

    print(f"Sucesso! Gerado:")
    print(f" - {output_graph_path}")

    return df_graph



In [ ]:
cases_path = Path('../../data/raw/cases.csv')
cases = pd.read_csv(cases_path)
used_cases = cases.iloc[[36]]

process_cases_to_graph_csvs(cases_df= used_cases, entity_classifier= entity_classifier, relation_classifier= relation_classifier)

Sucesso! Gerados:
 - nodes.csv: 35 nós registrados.
 - edges.csv: 19 arestas registradas.


(   node_id        case_id  node_type                        label_original  \
 0    N_001  PMC5137649_01    PATIENT                   A 44-year-old woman   
 1    N_002  PMC5137649_01  PROCEDURE              her physical examination   
 2    N_003  PMC5137649_01    PATIENT                                   She   
 3    N_004  PMC5137649_01  PROCEDURE                              contrast   
 4    N_005  PMC5137649_01  PROCEDURE                   computed tomography   
 5    N_006  PMC5137649_01      VALUE                   a 6cm cystic lesion   
 6    N_007  PMC5137649_01    PATIENT                                   She   
 7    N_008  PMC5137649_01  PROCEDURE                               EUS-FNA   
 8    N_009  PMC5137649_01    FINDING         normal pancreatic echotexture   
 9    N_010  PMC5137649_01  DIAGNOSIS                                a cyst   
 10   N_011  PMC5137649_01  PROCEDURE                                  that   
 11   N_012  PMC5137649_01  PROCEDURE               